# ⚡ Workflow Engines: Snakemake & Nextflow

## Learning Objectives
- Understand workflow management systems
- Write reproducible pipelines with Snakemake
- Understand Nextflow syntax and concepts
- Compare approaches for different use cases

## Why Workflow Engines?

**Problems with shell scripts:**
- No automatic parallelization
- No dependency tracking (re-runs everything)
- Hard to scale to clusters/cloud
- Poor reproducibility

**Workflow engines provide:**
- Automatic parallelization
- Smart re-execution (only changed steps)
- Container support (Docker, Singularity)
- Cluster/cloud execution
- Provenance tracking

---
## Part 1: Snakemake

Python-based, Make-like syntax. Define rules with inputs, outputs, and commands.

In [ ]:
# Example Snakefile content
snakefile_content = '''
# Snakefile for RNA-seq analysis

SAMPLES = ["sample1", "sample2", "sample3"]

rule all:
    input:
        "results/counts_matrix.csv"

rule fastqc:
    input:
        "data/{sample}.fastq.gz"
    output:
        html="qc/{sample}_fastqc.html",
        zip="qc/{sample}_fastqc.zip"
    threads: 2
    shell:
        "fastqc {input} -o qc/ -t {threads}"

rule align:
    input:
        fastq="data/{sample}.fastq.gz",
        index="reference/genome.idx"
    output:
        bam="aligned/{sample}.bam"
    threads: 8
    shell:
        "STAR --genomeDir {input.index} "
        "--readFilesIn {input.fastq} "
        "--runThreadN {threads} "
        "--outFileNamePrefix aligned/{wildcards.sample}_ "
        "--outSAMtype BAM SortedByCoordinate"

rule count:
    input:
        bam="aligned/{sample}.bam",
        gtf="reference/genes.gtf"
    output:
        "counts/{sample}.txt"
    shell:
        "featureCounts -a {input.gtf} -o {output} {input.bam}"

rule merge_counts:
    input:
        expand("counts/{sample}.txt", sample=SAMPLES)
    output:
        "results/counts_matrix.csv"
    script:
        "scripts/merge_counts.py"
'''

print(snakefile_content)

### Snakemake Key Concepts

| Concept | Description |
|---------|-------------|
| `rule` | A processing step |
| `input` | Required files |
| `output` | Produced files |
| `wildcards` | Pattern matching `{sample}` |
| `expand()` | Generate file lists |
| `threads` | Parallel resources |
| `conda` | Environment specification |

In [ ]:
# Running Snakemake
commands = '''
# Dry run (see what would execute)
snakemake -n

# Run with 8 cores
snakemake --cores 8

# With conda environments
snakemake --cores 8 --use-conda

# Generate DAG visualization
snakemake --dag | dot -Tpng > dag.png

# Run on SLURM cluster
snakemake --cluster "sbatch -p normal" --jobs 100
'''
print(commands)

---
## Part 2: Nextflow

Groovy-based DSL, designed for scalable genomics pipelines. Channels and processes.

In [ ]:
# Example Nextflow pipeline
nextflow_content = '''
#!/usr/bin/env nextflow

// Parameters
params.reads = "data/*_{1,2}.fastq.gz"
params.genome = "reference/genome.fa"
params.outdir = "results"

// Create channels
Channel
    .fromFilePairs(params.reads)
    .set { read_pairs_ch }

// Process: Quality Control
process FASTQC {
    tag "$sample_id"
    publishDir "${params.outdir}/qc", mode: "copy"
    
    input:
    tuple val(sample_id), path(reads)
    
    output:
    path "*_fastqc.{html,zip}"
    
    script:
    """
    fastqc ${reads}
    """
}

// Process: Alignment
process ALIGN {
    tag "$sample_id"
    cpus 8
    memory "32 GB"
    
    input:
    tuple val(sample_id), path(reads)
    path genome
    
    output:
    tuple val(sample_id), path("${sample_id}.bam")
    
    script:
    """
    bwa mem -t ${task.cpus} ${genome} ${reads} | \
    samtools sort -o ${sample_id}.bam
    """
}

// Workflow definition
workflow {
    FASTQC(read_pairs_ch)
    ALIGN(read_pairs_ch, params.genome)
}
'''

print(nextflow_content)

### Nextflow Key Concepts

| Concept | Description |
|---------|-------------|
| `Channel` | Data streams between processes |
| `process` | A computation unit |
| `workflow` | Connects processes |
| `tuple` | Grouped data (id + files) |
| `publishDir` | Copy outputs to results |
| `cpus/memory` | Resource requirements |

In [ ]:
# Running Nextflow
commands = '''
# Run pipeline
nextflow run main.nf

# With custom parameters
nextflow run main.nf --reads "data/*.fastq.gz" --outdir results

# With Docker containers
nextflow run main.nf -with-docker

# Resume interrupted run
nextflow run main.nf -resume

# Run from GitHub
nextflow run nf-core/rnaseq -r 3.12.0 --input samplesheet.csv
'''
print(commands)

---
## Part 3: Comparison

| Feature | Snakemake | Nextflow |
|---------|-----------|----------|
| Language | Python | Groovy/DSL |
| Learning curve | Easier | Steeper |
| File-based | Yes (Make-like) | Channel-based |
| Cloud support | Good | Excellent |
| nf-core ecosystem | - | Rich library |
| Best for | Custom pipelines | Production genomics |

## 🧬 Exercise: Design a Variant Calling Pipeline

Sketch a workflow (Snakemake or Nextflow) for:
1. FastQC → Trim reads → Align (BWA) → Mark duplicates → Call variants (GATK)

In [ ]:
# Your sketch here
# rule trim_reads:
#     ...
# rule align:
#     ...

## Summary

**Workflow engines** automate bioinformatics pipelines with:
- Reproducibility (containers, environments)
- Scalability (local → cluster → cloud)
- Efficiency (parallelization, smart re-runs)

**Recommendations:**
- Start with **Snakemake** for learning
- Use **nf-core** pipelines for standard analyses
- Build custom pipelines when needed